In [0]:
dbutils.widgets.text("input_folder", "Hackathon2/", "Input DICOM Folder")

In [0]:
import pydicom
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import time
import copy
import os
from io import BytesIO
from pydicom.filebase import DicomFileLike
from openai import OpenAI
from PIL import Image
import base64



In [0]:
input_dir = f"/Volumes/1_inland/sectra/vone/{dbutils.widgets.get("input_folder")}"


def list_dcm_files(folder):
    files = os.listdir(folder)
    files = [x for x in files if x.endswith(".dcm")  and "DICOMDIR" not in x]
    return files


df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .option("includeBinaryFiles", "false")
    .load(input_dir)
    .select("path")
)
print(df.count())
display(df.limit(10))


In [0]:
import pandas as pd
from pathlib import Path

api_key = dbutils.secrets.get(scope = "adc_store", key = "hackathon_gpt41mini_api_key")

def llm_pii_detection_batch(pdf_iter):
    system_prompt = "Output the result in a structured format with 3 required fields: is_text_detected, is_personal_information_detected, and confidence_score. Valid values for is_text_detected are yes, no, and not_sure. Valid values for is_personal_information_detected are yes, no, and not_sure. confidence_score is an integer from 0 to 100. 0 indicates high confidence of no text detected. 100 indicates high confidence of text being detected. 50 indicates uncertain. Do not leave any field empty. Always respond with 3 fields everytime."

    user_prompt = "Tell me if there is any text in the image. Also tell me if these texts contain any personal information e.g. name, date of birth, scan date, ID numbers. Then give me a confidence score."

    deployment_name = "gpt-4.1-mini"

    endpoint = "https://telefonica-hackathon-2026.cognitiveservices.azure.com/openai/v1/"



    client = OpenAI(base_url=f"{endpoint}",api_key=api_key)

    for pdf in pdf_iter:
        redacted_rows = []
        #for path, content in zip(pdf['path'], pdf['content']):
        for path in pdf['path']:
            try:
                is_loaded_ok = False

                '''
                #ds = pydicom.dcmread(BytesIO(content), force=True)
                ds = pydicom.dcmread(path[5:])
                #ds.decompress()


                # Extract pixel array
                pixel_array = ds.pixel_array.astype(np.float32)

                # Normalize to 0-255
                pixel_array -= pixel_array.min()
                pixel_array /= pixel_array.max()
                pixel_array *= 255.0
                pixel_array = pixel_array.astype(np.uint8)

                # Convert to PIL image
                image = Image.fromarray(pixel_array)

                # Save PNG to memory buffer
                buffer = BytesIO()
                image.save(buffer, format="PNG")

                # Convert to Base64
                base64_str = base64.b64encode(buffer.getvalue()).decode("utf-8")
                '''

                ds = pydicom.dcmread(path[5:], stop_before_pixels=False)

                # Get pixel data (no extra copy yet)
                arr = ds.pixel_array  # already numpy

                # Normalize WITHOUT converting to float32 full-size
                arr_min = arr.min()
                arr_max = arr.max()

                if arr_max > arr_min:
                    # Use float32 temporarily but avoid multiple copies
                    arr = ((arr - arr_min) * (255.0 / (arr_max - arr_min))).astype(np.uint8)
                else:
                    arr = np.zeros(arr.shape, dtype=np.uint8)

                # Convert directly to image
                image = Image.fromarray(arr)

                # Encode directly to base64 (avoid buffer.getvalue() copy)
                buffer = BytesIO()
                image.save(buffer, format="PNG", optimize=True)

                base64_str = base64.b64encode(buffer.getbuffer()).decode("utf-8")

                # Explicit cleanup (important in UDFs)
                buffer.close()
                del arr, image

                is_loaded_ok = True




                completion = client.chat.completions.create(
                    model=deployment_name,
                    messages=[
                        {
                            "role": "system",
                            "content": system_prompt,
                        },
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text", "text": user_prompt
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/png;base64,{base64_str}"}
                                }

                            ]
                        }
                    ]
                )

                result = completion.choices[0].message.content
                
                redacted_rows.append((path, is_loaded_ok, result, None))
            except Exception as e:
                redacted_rows.append((path, is_loaded_ok, None, str(e)[:1000]))


        yield pd.DataFrame(redacted_rows, columns=["input_path", "is_loaded_ok", "llm_result", "error_message"])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BinaryType, BooleanType

# Define output schema: path + redacted DICOM binary
output_schema = StructType([
    StructField("input_path", StringType(), True),
    StructField("is_loaded_ok", BooleanType(), True),
    StructField("llm_result", StringType(), True),
    StructField("error_message", StringType(), True)
])

# Run mapInPandas redaction
pii_detect_df = df.mapInPandas( llm_pii_detection_batch, schema=output_schema)

In [0]:
#results = pii_detect_df.collect()
pii_detect_df = pii_detect_df.repartition(1000)
pii_detect_df.write.mode("overwrite").parquet(f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get("input_folder")).replace("/", "")}_output.parquet")


In [0]:
output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('input_folder')).replace('/', '')}_output.parquet"
df_out = spark.read.parquet(output_path)
display(df_out.limit(1000))